# Land skill, soil-moisture memory and land-atmosphere coupling

## Purpose

Does initializing the land with DART (DA_LND, DA_COUPLED) improve the forecast soil moisture and near-surface temperature over land, and how long does the initial soil-moisture anomaly persist and couple to the atmosphere in each configuration (ESP-Lab `4a_soil_moisture_memory`)?

## Inputs

ELM daily `SOILWATER_10CM`, `H2OSOI` (top 5 cm, layer-thickness weighted, `lnd_sea_mask/dzsoi_elm.nc`), `TSA`; EAM 6-hourly `LHFLX`. References: ESA CCI daily soil moisture (`ESA_CCI_H2OSOI/1x1/daily`) and ERA5 2-m temperature. Model climatology: monthly `SOILLIQ`/`SOILICE` (top 5 cm as volumetric water) and `TSA` of the free-running E3SMv3 AMIP runs `AMIP_CASES`.

## Method

1. Lead-day ensemble performance over land regions (`util.fcst_init_skill.lead_scores`) of `SOILM_5CM` vs ESA CCI (on the days and points ESA CCI retrieves) and `TSA` vs ERA5 TREFHT.
2. Weekly anomaly skill as in notebook 02: observed anomalies from the monthly reference climatology of `CLIM_YEARS` (monthly means of the daily ESA CCI and ERA5 `1x1/daily` TREFHT); forecast anomalies both ways of `BIASES`, against the observed climatology (`"none"`: drift and model bias count as error) and against the AMIP monthly climatology of the same years (`"amip"`: model mean bias removed). Their difference (`land_bias_impact_*`) shows the effect of the model mean bias on the land skill.
3. Ensemble-based memory and coupling (`util.fcst_init_skill.memory_and_coupling`): weekly anomalies of each member from the mean over members and starts of that week; at each land point, over the 80 member-starts of a season, the correlation of week-1 soil moisture with week w (memory) and the same-week correlation of soil moisture with TSA and LHFLX (coupling; negative SM-TSA, positive SM-LHFLX where evaporation is soil-moisture limited).

## Outputs

`data/hind_lead_eval/land/`: `lead_skill_<CONFIG>_<SEASON>_<VAR>.nc`, `anom_<BIAS>_<CONFIG>_<SEASON>_<VAR>.nc`, `land_weekly_scores.csv` (per bias treatment), `memory_coupling_<CONFIG>_<SEASON>.nc`, `memory_coupling_regional.csv`; figures in `figure/hind_lead_eval/land/` (weekly skill per bias treatment, `land_bias_impact_<SEASON>.png`, memory and coupling maps and curves).

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction, ELM layer thicknesses
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py) and seasons (4 starts each)
CONFIGS = ("CLIM_BASE", "HYB_BASE", "DA_ATM", "DA_LND", "DA_COUPLED")
SEASONS = ("DJF", "JJA")
MAX_LEAD = 56          # lead days (8 weeks)
MAX_MEMBERS = None     # e.g. 4 for a quick test
REQUIRE_ALL_MEMBERS = True  # skip a (configuration, season) until every member's stream is regenerated

SKILL_VARIABLES = ("SOILM_5CM", "TSA")
SM_VARIABLE = "SOILWATER_10CM"
COUPLED_VARIABLES = ("TSA", "LHFLX")
REGIONS = ("LAND", "CONUS")
CLIM_YEARS = (2001, 2014)  # common to ERA5, GPCP V3.3 (from 1998), NOAA OLR and the AMIP monthly output (FLUT to 2014)
BIASES = ("none", "amip")  # forecast anomaly vs the observed climatology ("none": drift and model bias count as error)
                           # and vs the AMIP climatology ("amip": model mean bias removed); "loo" for comparison only
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                 # free-running E3SMv3 DECK AMIP runs
AMIP_CASES = ("v3.LR.amip_0101", "v3.LR.amip_0151")
BASELINE = "HYB_BASE"
MAP_WEEKS = (2, 4)

FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 200

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import MEMBERS, check_configs, init_dates, readiness
from configs.output_paths import workflow_output_dirs

check_configs(CONFIGS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "land", output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(data_root=INPUT_DATA_ROOT, land_support_dir=LAND_SUPPORT_DIR)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")


def members_of(config):
    return MEMBERS[config] if MAX_MEMBERS is None else MEMBERS[config][:MAX_MEMBERS]


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

In [ ]:
status = readiness(CONFIGS, SEASONS, streams=(("lnd", "daily"), ("atm", "6hourly")), data_root=INPUT_DATA_ROOT)
summary = status.groupby(["config", "season"]).agg(ready=("ready", "sum"), members=("members", "sum"))
display(summary.unstack("season"))
RUNS = [(c, s) for (c, s), row in summary.iterrows() if row.ready >= (row.members if REQUIRE_ALL_MEMBERS else 1)]
print(f"[RUNS] {len(RUNS)} of {len(summary)} configuration-seasons ready: {RUNS}")

## 5. Derive and save

Lead-day scores and weekly anomalies (for every treatment in `BIASES`, one read of each ensemble) of the verified land variables, then the memory and coupling correlations. Saved files are reused unless `FORCE_COMPUTE`.

In [ ]:
from configs.s2s_config import TCC_REGIONS
from util.fcst_init_data import model_climatology, open_campaign, open_reference, reference_climatology, region_masks, spec
from util.fcst_init_skill import lead_scores, memory_and_coupling, weekly_anomalies

regions = [r for r in TCC_REGIONS if r["name"] in REGIONS]
years = list(range(CLIM_YEARS[0], CLIM_YEARS[1] + 1))
lead_path = lambda c, s, v: data_dir / f"lead_skill_{c}_{s}_{v}.nc"
anom_path = lambda b, c, s, v: data_dir / f"anom_{b}_{c}_{s}_{v}.nc"
mc_path = lambda c, s: data_dir / f"memory_coupling_{c}_{s}.nc"


def cached_climatology(path, build):
    """Saved monthly-of-daily climatology of the current ``CLIM_YEARS``, else built and saved."""
    if path.exists() and not FORCE_COMPUTE:
        clim = xr.load_dataarray(path)
        source = clim.attrs.get("climatology_source", "")
        if clim.attrs.get("climatology_years") == f"{years[0]}-{years[-1]}" and (
                source.startswith("E3SMv3 AMIP") or "monthly means of the daily series" in source):
            return clim
    clim = build()
    clim.to_netcdf(path)
    print(f"[WRITE] {path}  ({clim.attrs.get('climatology_source', '')})")
    return clim


def anom_current(path, bias):
    """An anomaly file made with ``bias`` and the current ``CLIM_YEARS``."""
    if not path.exists():
        return False
    with xr.open_dataset(path) as ds:
        return ds.attrs.get("bias") == bias and ds.attrs.get("clim_years") == str(CLIM_YEARS)

masks = None
for season in SEASONS:
    inits = init_dates([season])[season]
    for var in SKILL_VARIABLES:
        # per configuration: (lead scores needed, bias treatments whose anomalies are needed)
        todo = {c: (FORCE_COMPUTE or not lead_path(c, season, var).exists(),
                    [b for b in BIASES if FORCE_COMPUTE or not anom_current(anom_path(b, c, season, var), b)])
                for c in CONFIGS if (c, season) in RUNS}
        todo = {c: t for c, t in todo.items() if t[0] or t[1]}
        if not todo:
            continue
        ref = open_reference(var, inits, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT).load()
        clim = cached_climatology(data_dir / f"clim_{season}_{var}.nc", lambda: reference_climatology(
            var, inits, years, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT))
        mclim = cached_climatology(data_dir / f"clim_model_{season}_{var}.nc", lambda: model_climatology(
            var, inits, years, max_lead=MAX_LEAD, amip_root=AMIP_ROOT, cases=AMIP_CASES,
            land_support_dir=LAND_SUPPORT_DIR)) if any("amip" in bs for _, bs in todo.values()) else None
        if masks is None:
            masks = region_masks(regions, ref.lat.values, ref.lon.values, land_support_dir=LAND_SUPPORT_DIR)
        land = masks.sel(region="LAND", drop=True)
        for c, (need_lead, biases) in todo.items():
            model = open_campaign(c, season, var, members=members_of(c), max_lead=MAX_LEAD, **ROOTS).load()
            if need_lead:
                lead_scores(model.where(land), ref.where(land), masks).assign_attrs(units=spec(var).units).to_netcdf(lead_path(c, season, var))
                print(f"[WRITE] {lead_path(c, season, var)}")
            for b in biases:  # one read of the ensemble for every bias treatment
                fa, oa = weekly_anomalies(model, ref.where(land), clim, bias=b, model_climatology=mclim)
                xr.Dataset({"model": fa, "reference": oa}).assign_attrs(
                    bias=b, clim_years=str(CLIM_YEARS), reference_climatology=clim.attrs.get("climatology_source", ""),
                    model_climatology=mclim.attrs.get("climatology_source", "") if b == "amip" else "",
                ).to_netcdf(anom_path(b, c, season, var))
                print(f"[WRITE] {anom_path(b, c, season, var)}")
            del model
    for c in CONFIGS:
        if (c, season) not in RUNS or (mc_path(c, season).exists() and not FORCE_COMPUTE):
            continue
        sm = open_campaign(c, season, SM_VARIABLE, members=members_of(c), max_lead=MAX_LEAD, **ROOTS).load()
        other = {v: open_campaign(c, season, v, members=members_of(c), max_lead=MAX_LEAD, **ROOTS).load()
                 for v in COUPLED_VARIABLES}
        memory_and_coupling(sm, other).to_netcdf(mc_path(c, season))
        print(f"[WRITE] {mc_path(c, season)}")
        del sm, other

lead = {(c, s, v): xr.load_dataset(lead_path(c, s, v)) for c in CONFIGS for s in SEASONS for v in SKILL_VARIABLES
        if lead_path(c, s, v).exists()}
anom = {(b, c, s, v): xr.load_dataset(anom_path(b, c, s, v))
        for b in BIASES for c in CONFIGS for s in SEASONS for v in SKILL_VARIABLES if anom_path(b, c, s, v).exists()}
mc = {(c, s): xr.load_dataset(mc_path(c, s)) for c in CONFIGS for s in SEASONS if mc_path(c, s).exists()}
print(f"[LOAD] {len(lead)} lead-skill, {len(anom)} anomaly, {len(mc)} memory/coupling files")

## 6. Scores

Weekly pattern ACC and MSSS over land regions; regional means (land points) of the memory and coupling correlations.

In [ ]:
from util.fcst_init_skill import msss, pattern_acc

if masks is None and anom:
    d = next(iter(anom.values()))
    masks = region_masks(regions, d.lat.values, d.lon.values, land_support_dir=LAND_SUPPORT_DIR)
rows = []
for (b, c, s, v), ds in anom.items():
    pa = pattern_acc(ds.model, ds.reference, masks).mean("init")
    sk = msss(ds.model, ds.reference, masks)
    for r in masks.region.values:
        for w in ds.week.values:
            rows.append(dict(bias=b, config=c, season=s, variable=v, region=r, week=int(w), pattern_acc=float(pa.sel(region=r, week=w)),
                             msss=float(sk.msss.sel(region=r, week=w)), rmse_anom=float(sk.rmse_anom.sel(region=r, week=w))))
weekly = pd.DataFrame(rows)
weekly.to_csv(data_dir / "land_weekly_scores.csv", index=False)
if len(weekly):
    display(weekly[weekly.region == "LAND"].pivot_table(index=["variable", "config"], columns=["bias", "season", "week"],
                                                        values="pattern_acc", sort=False).round(2))

rows = []
cos = None
for (c, s), ds in mc.items():
    if cos is None:
        cos = np.cos(np.deg2rad(ds.lat))
    for r in masks.region.values:
        w = (cos * masks.sel(region=r)).where(ds.memory.notnull())
        for name in ds.data_vars:
            series = (ds[name] * w).sum(("lat", "lon")) / w.sum(("lat", "lon"))
            rows += [dict(config=c, season=s, region=r, metric=name, week=int(k), value=float(series.sel(week=k)))
                     for k in ds.week.values]
regional = pd.DataFrame(rows)
regional.to_csv(data_dir / "memory_coupling_regional.csv", index=False)
if len(regional):
    display(regional[regional.region == "LAND"].pivot_table(index=["metric", "config"], columns=["season", "week"],
                                                            values="value", sort=False).round(2))

## 7. Plot and show

In [ ]:
from util.fcst_init_plotting import plot_bias_impact, plot_config_week_maps, plot_lead_curves, plot_weekly_regional

for s in SEASONS:
    sub = {(c, v): ds for (c, ss, v), ds in lead.items() if ss == s}
    if sub:
        save(plot_lead_curves(sub, SKILL_VARIABLES, region="LAND", metrics=("rmse", "bias", "spread_ratio"),
                              max_lead=MAX_LEAD, season=s), f"land_lead_skill_{s}.png")
    for b in BIASES:
        wb = weekly[(weekly.bias == b) & (weekly.season == s)] if len(weekly) else weekly
        if len(wb):
            fig = plot_weekly_regional(wb, SKILL_VARIABLES, region="LAND", season=s)
            fig.suptitle(f"{s}: weekly anomaly skill, LAND (BIAS = {b})")
            save(fig, f"land_weekly_{b}_{s}.png")
    if {"none", "amip"} <= set(BIASES) and len(weekly) and len(weekly[weekly.season == s]):
        save(plot_bias_impact(weekly, SKILL_VARIABLES, region="LAND", season=s), f"land_bias_impact_{s}.png")
    maps = {c: ds for (c, ss), ds in mc.items() if ss == s}
    if not maps:
        continue
    for name, label in (("memory", "r(SM week 1, SM week w)"),
                        *((f"coupling_{v}", f"r(SM, {v})") for v in COUPLED_VARIABLES)):
        for w in MAP_WEEKS:
            save(plot_config_week_maps({c: ds[name] for c, ds in maps.items()}, week=w, label=label,
                                       title=f"{s} {label}, week {w} (members x starts)"), f"land_{name}_{s}_week{w}.png")
    fig, axes = plt.subplots(1, 1 + len(COUPLED_VARIABLES), figsize=(5 * (1 + len(COUPLED_VARIABLES)), 3.6),
                             constrained_layout=True)
    from configs.fcst_init_experiment_config import STYLE_MAP
    for ax, name in zip(np.atleast_1d(axes), ["memory"] + [f"coupling_{v}" for v in COUPLED_VARIABLES]):
        t = regional[(regional.season == s) & (regional.region == "LAND") & (regional.metric == name)]
        for c, g in t.groupby("config", sort=False):
            ax.plot(g.week, g.value, marker="o", label=c, **STYLE_MAP.get(c, {}))
        ax.axhline(0, color="k", linewidth=0.6)
        ax.set_title(f"{name} (land mean)", loc="left")
        ax.set_xlabel("Lead week")
    np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
    save(fig, f"land_memory_coupling_{s}.png")